# ETL — Datos de Empresas IBEX 35


**Fase:** Ingeniería del Dato  

---

En este cuaderno voy a realizar el proceso ETL de los datos de las 35 empresas del IBEX 35 y el archivo referencia de empresas:
1. **Extracción**: lectura de los archivos `.xlsx` en bruto
2. **Transformación**: limpieza, estandarización y cálculo de variables derivadas
3. **Carga**: almacenamiento en la base de datos SQLite `tfg.db`

## 1. Importación de librerías y configuración de rutas

In [4]:
import pandas as pd
import numpy as np
import os
import sqlite3
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# ── Rutas del proyecto ──────────────────────────────────────────────────────
BASE        = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG')
DATOS_DIR   = os.path.join(BASE, 'Datos', 'Empresas')
DB_PATH     = os.path.join(BASE, 'proyecto', 'data', 'db', 'tfg.db')
PROC_DIR    = os.path.join(BASE, 'proyecto', 'data', 'processed')

# Conexión a la base de datos
engine = create_engine(f'sqlite:///{DB_PATH}')

print('Rutas configuradas correctamente')
print(f'  Datos brutos : {DATOS_DIR}')
print(f'  Base de datos: {DB_PATH}')

Rutas configuradas correctamente
  Datos brutos : /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/Datos/Empresas
  Base de datos: /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/proyecto/data/db/tfg.db


## 2. Lectura del fichero de referencia 

En esta celda voy a realizar la lectura del fichero referencia de empresas                                                                
Este fichero contiene el listado de las 35 empresas con su ticker, ISIN y sector GICS.

In [5]:
# Leer fichero de referencia GridExport (ticker, ISIN, sector)                                                                                                                                        
grid_file = os.path.join(DATOS_DIR, 'GridExport_November_5_2025_21_27_3.xlsx')
df_ref = pd.read_excel(grid_file)
df_ref = df_ref[['Identifier (RIC)', 'Company Common Name', 'Ticker Symbol', 'GICS Sector Name', 'ISIN']].copy()
df_ref.columns = ['ric', 'empresa', 'ticker', 'sector', 'isin']
df_ref = df_ref.dropna(subset=['ric'])
df_ref = df_ref[df_ref['ric'] != '.IBEX'].reset_index(drop=True)

print(f'Empresas encontradas: {len(df_ref)}')
df_ref

Empresas encontradas: 36


,ric,empresa,ticker,sector,isin
0,Totals (35),NaN,NaN,NaN,NaN
1,ROVI.MC,Laboratorios Farmaceuticos ROVI SA,ROVI,Health Care,ES0157261019
2,SCYR.MC,Sacyr SA,SCYR,Industrials,ES0182870214
3,BKT.MC,Bankinter SA,BKT,Financials,ES0113679I37
4,AMA.MC,Amadeus IT Group SA,AMS,Consumer Discretionary,ES0109067019
5,REP.MC,Repsol SA,REP,Energy,ES0173516115
6,SLRS.MC,Solaria Energia y Medio Ambiente SA,SLR,Utilities,ES0165386014
7,ACX.MC,Acerinox SA,ACX,Materials,ES0132105018
8,SAN.MC,Banco Santander SA,SAN,Financials,ES0113900J37
9,AENA.MC,Aena SME SA,AENA,Industrials,ES0105046017


## 3. Función de lectura y limpieza de archivos de empresa

Cada archivo `.xlsx` de cada empresa tiene una cabecera de metadata hasta la fila 30. Los datos reales que nos interesan empiezan en la fila 31 con las columnas: `Exchange Date, Close, Net, %Chg, Open, Low, High, Volume, Turnover EUR`.

In [6]:
def leer_empresa(filepath, ticker):
    """
    Lee y limpia el archivo Excel de precios de una empresa del IBEX 35.
    
    Los archivos tienen metadata en las primeras filas. La función localiza
    automáticamente la fila de cabecera buscando 'Exchange Date'.
    """
    # Leer sin cabecera para buscar dónde empiezan los datos
    df_raw = pd.read_excel(filepath, header=None)
    
    # Buscar la fila que contiene 'Exchange Date'
    header_row = None
    for i, row in df_raw.iterrows():
        if any('exchange date' in str(v).lower() for v in row.values):
            header_row = i
            break
    
    if header_row is None:
        print(f'  AVISO: No se encontró cabecera en {ticker}')
        return None
    
    # Releer desde la fila de cabecera
    df = pd.read_excel(filepath, header=header_row)
    
    # Renombrar columnas a nombres estándar
    col_map = {
        'Exchange Date' : 'fecha',
        'Close'         : 'close',
        'Net'           : 'net',
        '%Chg'          : 'pct_chg',
        'Open'          : 'open',
        'Low'           : 'low',
        'High'          : 'high',
        'Volume'        : 'volume',
        'Turnover EUR'  : 'turnover_eur',
        'Unnamed: 9'    : 'flujo'
    }
    df = df.rename(columns={c: col_map[c] for c in df.columns if c in col_map})
    
    # Seleccionar solo las columnas que nos interesan
    cols_utiles = [c for c in ['fecha','close','net','pct_chg','open','low','high','volume','turnover_eur','flujo'] if c in df.columns]
    df = df[cols_utiles].copy()
    
    # Parsear fechas y eliminar filas sin fecha válida
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
    df = df.dropna(subset=['fecha'])
    
    # Filtrar desde 2005 en adelante
    df = df[df['fecha'] >= '2005-01-01'].copy()
    
    # Convertir columnas numéricas
    for col in ['close','net','pct_chg','open','low','high','volume','turnover_eur','flujo']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Añadir ticker e identificador
    df['ticker'] = ticker
    
    # Ordenar por fecha ascendente
    df = df.sort_values('fecha').reset_index(drop=True)
    
    return df

print('Función de lectura definida correctamente')

Función de lectura definida correctamente


## 4. Procesamiento de todas las empresas

Iteramos sobre todos los archivos `.xlsx` de las empresas que los tengo guardados dentro de la carpeta `Empresas`,pero excluyo los que continenen los datos de referencia. Aplicamos la función de limpieza para consolidar los datos en un único DataFrame en la siguiente celda.

In [7]:
# Archivos a excluir (no son datos de precios)
EXCLUIR = ['RIC.xlsx', 'GridExport_November_5_2025_21_27_3.xlsx']

archivos = sorted([
    f for f in os.listdir(DATOS_DIR)
    if f.endswith('.xlsx') and not f.startswith('~$') and f not in EXCLUIR
])

print(f'Archivos a procesar: {len(archivos)}')
print('─' * 50)

dfs = []
resumen = []

for archivo in archivos:
    ticker = archivo.split('_')[0]  # Extraer ticker del nombre del archivo
    filepath = os.path.join(DATOS_DIR, archivo)
    
    df = leer_empresa(filepath, ticker)
    
    if df is not None and len(df) > 0:
        dfs.append(df)
        nulos = df.isnull().sum()['close']
        resumen.append({
            'ticker'   : ticker,
            'filas'    : len(df),
            'desde'    : df['fecha'].min().date(),
            'hasta'    : df['fecha'].max().date(),
            'nulos_close': nulos
        })
        print(f'  OK  {ticker:<12} {len(df):>5} filas  {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    else:
        print(f'  ERROR {ticker}')

print('─' * 50)
print(f'Total empresas procesadas: {len(dfs)}')

Archivos a procesar: 35
──────────────────────────────────────────────────
  OK  ACS.MC        5323 filas  2005-01-03 → 2025-10-31
  OK  ACX.MC        5323 filas  2005-01-03 → 2025-10-31
  OK  AENA.MC       2745 filas  2015-02-11 → 2025-10-31
  OK  AMA.MC        3971 filas  2010-04-29 → 2025-10-31
  OK  ANA.MC        5323 filas  2005-01-03 → 2025-10-31
  OK  ANE.MC        1112 filas  2021-07-01 → 2025-10-31
  OK  BBVA.MC       5323 filas  2005-01-03 → 2025-10-31
  OK  BKT.MC        5323 filas  2005-01-03 → 2025-10-31
  OK  CABK.MC       4615 filas  2007-10-10 → 2025-10-31
  OK  CLNX.MC       2687 filas  2015-05-07 → 2025-10-31
  OK  COL.MC        5322 filas  2005-01-03 → 2025-10-31
  OK  ELE.MC        5323 filas  2005-01-03 → 2025-10-31
  OK  ENAG.MC       5323 filas  2005-01-03 → 2025-10-31
  OK  FER.MC        5322 filas  2005-01-03 → 2025-10-31
  OK  FLUI.MC       4600 filas  2007-10-31 → 2025-10-31
  OK  GRLS.MC       4974 filas  2006-05-17 → 2025-10-31
  OK  IBE.MC        5323 fila

## 5. Consolidación y resumen del dataset

In [8]:
# Unir todos los DataFrames en uno solo
df_empresas = pd.concat(dfs, ignore_index=True)

print(f'Dataset consolidado: {df_empresas.shape[0]:,} filas x {df_empresas.shape[1]} columnas')
print(f'Período total: {df_empresas["fecha"].min().date()} → {df_empresas["fecha"].max().date()}')
print(f'Empresas: {df_empresas["ticker"].nunique()}')
print()

# Resumen por empresa
df_resumen = pd.DataFrame(resumen)
print('Resumen por empresa:')
df_resumen

Dataset consolidado: 157,455 filas x 9 columnas
Período total: 2005-01-03 → 2025-10-31
Empresas: 35

Resumen por empresa:


,ticker,filas,desde,hasta,nulos_close
0,ACS.MC,5323,2005-01-03,2025-10-31,0
1,ACX.MC,5323,2005-01-03,2025-10-31,0
2,AENA.MC,2745,2015-02-11,2025-10-31,0
3,AMA.MC,3971,2010-04-29,2025-10-31,0
4,ANA.MC,5323,2005-01-03,2025-10-31,0
5,ANE.MC,1112,2021-07-01,2025-10-31,0
6,BBVA.MC,5323,2005-01-03,2025-10-31,0
7,BKT.MC,5323,2005-01-03,2025-10-31,0
8,CABK.MC,4615,2007-10-10,2025-10-31,1
9,CLNX.MC,2687,2015-05-07,2025-10-31,0


## 6. Cálculo de variables derivadas

Calculamos el **log-retorno diario** y la **volatilidad histórica** con ventana móvil de 21 días (aproximadamente 1 mes de trading). El **log-retorno diario** lo calculo porque tiene mejores propiedades que los precios y es mas cercano a una distribución normal. La **volatilidad histórica** con ventana móvil de 21 días la calculo ya que (R_total ≈ Σ r_diarios), evitamos el ruido diario, es el estándar que se usa en finanzas y me permite facilmente relacionarla con los cambios macro.

In [9]:
# Calcular log-retorno y volatilidad histórica por empresa
df_empresas = df_empresas.sort_values(['ticker', 'fecha']).reset_index(drop=True)

# Log-retorno diario
df_empresas['log_ret'] = df_empresas.groupby('ticker')['close'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Volatilidad histórica anualizada (ventana 21 días)
df_empresas['vol_hist_21d'] = df_empresas.groupby('ticker')['log_ret'].transform(
    lambda x: x.rolling(window=21, min_periods=10).std() * np.sqrt(252)
)

print('Variables derivadas calculadas:')
print(f'  log_ret      : log-retorno diario')
print(f'  vol_hist_21d : volatilidad histórica anualizada (ventana 21 días)')
print()
print('Primeras filas de ejemplo (Santander):')
df_empresas[df_empresas['ticker'] == 'SAN.MC'][['fecha','close','log_ret','vol_hist_21d']].head(10)

Variables derivadas calculadas:
  log_ret      : log-retorno diario
  vol_hist_21d : volatilidad histórica anualizada (ventana 21 días)

Primeras filas de ejemplo (Santander):


,fecha,close,log_ret,vol_hist_21d
134656,2005-01-03,5.462551,NaN,NaN
134657,2005-01-04,5.462551,0.000000,NaN
134658,2005-01-05,5.385362,-0.014231,NaN
134659,2005-01-07,5.361612,-0.004420,NaN
134660,2005-01-10,5.349737,-0.002217,NaN
134661,2005-01-11,5.355675,0.001109,NaN
134662,2005-01-12,5.343799,-0.002220,NaN
134663,2005-01-13,5.397237,0.009950,NaN
134664,2005-01-14,5.397237,0.000000,NaN
134665,2005-01-17,5.468488,0.013115,NaN


In [10]:
df_empresas[df_empresas['ticker'] == 'SAN.MC'][['fecha','close','log_ret','vol_hist_21d']].dropna().head(5)

,fecha,close,log_ret,vol_hist_21d
134666,2005-01-18,5.456613,-0.002174,0.119354
134667,2005-01-19,5.444738,-0.002179,0.113662
134668,2005-01-20,5.403175,-0.007663,0.113507
134669,2005-01-21,5.355675,-0.008830,0.114131
134670,2005-01-24,5.343799,-0.002220,0.109694


## 7. Control de calidad

Revisamos valores nulos, duplicados y rango de fechas antes de cargar en la base de datos.

In [11]:
print('=== CONTROL DE CALIDAD ===')
print()

# Valores nulos por columna
nulos = df_empresas.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0])
print()

# Duplicados
dupl = df_empresas.duplicated(subset=['ticker', 'fecha']).sum()
print(f'Filas duplicadas (ticker + fecha): {dupl}')
print()

# Precios negativos o cero
precios_invalidos = (df_empresas['close'] <= 0).sum()
print(f'Precios <= 0: {precios_invalidos}')
print()

# Cobertura temporal por empresa
cobertura = df_empresas.groupby('ticker').agg(
    desde=('fecha', 'min'),
    hasta=('fecha', 'max'),
    dias=('fecha', 'count'),
    nulos_close=('close', lambda x: x.isnull().sum())
).reset_index()
print('Cobertura temporal por empresa:')
cobertura

=== CONTROL DE CALIDAD ===

Valores nulos por columna:
close             8
net              43
pct_chg          43
open             12
low              19
high             18
volume           18
log_ret          50
vol_hist_21d    351
dtype: int64

Filas duplicadas (ticker + fecha): 0

Precios <= 0: 0

Cobertura temporal por empresa:


,ticker,desde,hasta,dias,nulos_close
0,ACS.MC,2005-01-03,2025-10-31,5323,0
1,ACX.MC,2005-01-03,2025-10-31,5323,0
2,AENA.MC,2015-02-11,2025-10-31,2745,0
3,AMA.MC,2010-04-29,2025-10-31,3971,0
4,ANA.MC,2005-01-03,2025-10-31,5323,0
5,ANE.MC,2021-07-01,2025-10-31,1112,0
6,BBVA.MC,2005-01-03,2025-10-31,5323,0
7,BKT.MC,2005-01-03,2025-10-31,5323,0
8,CABK.MC,2007-10-10,2025-10-31,4615,1
9,CLNX.MC,2015-05-07,2025-10-31,2687,0


---
Revisando los valores nulos encontrados en la celda anterior decido hacer una imputación de nulos en close mediante interpolación lineal por empresa.
La interpolación lineal es el método estándar para series de precios ya que estima el precio faltante como el punto medio entre el anterior y el siguiente para no introducir autocorrelación artifical.

In [12]:
 # Imputación de nulos en close mediante interpolación lineal por empresa
  # La interpolación lineal es el método estándar para series de precios:
  # estima el precio faltante como el punto medio entre el anterior y el siguiente

df_empresas['close'] = df_empresas.groupby('ticker')['close'].transform(
      lambda x: x.interpolate(method='linear', limit_direction='both')
  )

  # Recalcular log_ret y vol_hist_21d tras la imputación
df_empresas['log_ret'] = df_empresas.groupby('ticker')['close'].transform(
      lambda x: np.log(x / x.shift(1))
  )
df_empresas['vol_hist_21d'] = df_empresas.groupby('ticker')['log_ret'].transform(
      lambda x: x.rolling(window=21, min_periods=10).std() * np.sqrt(252)
  )

  # Verificar que ya no quedan nulos en close
nulos_close = df_empresas['close'].isnull().sum()
print(f'Nulos en close tras imputación: {nulos_close}')

Nulos en close tras imputación: 0


## 8. Carga en la base de datos SQLite

Guardamos los datos en la tabla `precios_empresas` de la base de datos `tfg.db` que sera la base de datos en la que guarde todas las tablas. Guardo también un archivo backup en csv por si tengo problemas al leerlo con `pandas`

In [13]:
from sqlalchemy import Date                                                                                                                                                                           
   
  # Cargar en SQLite                                                                                                                                                                                    
df_empresas.to_sql(
      name      = 'precios_empresas',
      con       = engine,
      if_exists = 'replace',
      index     = False,
      dtype     = {'fecha': Date()}
  )

  # Cargar también la tabla de referencia
df_ref.to_sql(
      name      = 'ref_empresas',
      con       = engine,
      if_exists = 'replace',
      index     = False
  )

print('Tablas cargadas en tfg.db:')
print(f'  precios_empresas : {len(df_empresas):,} filas')
print(f'  ref_empresas     : {len(df_ref)} empresas')

  # Guardar también en Parquet como backup
parquet_path = os.path.join(PROC_DIR, 'precios_empresas.parquet')
try:
    df_empresas.to_parquet(parquet_path, index=False)
    print(f'  Parquet guardado : {parquet_path}')
except Exception as e:
    print(f'  Parquet no guardado (Arrow conflict): {e}')
    # Alternativa: guardar como CSV
    csv_path = os.path.join(PROC_DIR, 'precios_empresas.csv')
    df_empresas.to_csv(csv_path, index=False)
    print(f'  CSV guardado como alternativa: {csv_path}')


Tablas cargadas en tfg.db:
  precios_empresas : 157,455 filas
  ref_empresas     : 36 empresas
  Parquet guardado : /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/proyecto/data/processed/precios_empresas.parquet


In [14]:
# Guardar backup en CSV (más compatible que Parquet con pandas 3.0)                                                                                                                                   
csv_path = os.path.join(PROC_DIR, 'precios_empresas.csv')
df_empresas.to_csv(csv_path, index=False)
print(f'Backup CSV guardado: {csv_path}')

Backup CSV guardado: /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/proyecto/data/processed/precios_empresas.csv


## 9. Verificación final

Hago una consulta a la base de datos para confirmar que los datos se han cargado correctamente.

In [15]:
import sqlite3

# Verificar tablas en la BBDD
with sqlite3.connect(DB_PATH) as conn:
    tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
    print('Tablas en tfg.db:')
    print(tablas)
    print()
    
    # Contar filas por tabla
    n_precios = pd.read_sql("SELECT COUNT(*) as n FROM precios_empresas", conn).iloc[0,0]
    n_tickers = pd.read_sql("SELECT COUNT(DISTINCT ticker) as n FROM precios_empresas", conn).iloc[0,0]
    print(f'precios_empresas: {n_precios:,} filas, {n_tickers} empresas')
    print()
    
    # Muestra de datos
    muestra = pd.read_sql("""
        SELECT ticker, fecha, close, log_ret, vol_hist_21d
        FROM precios_empresas
        WHERE ticker = 'SAN.MC'
        ORDER BY fecha DESC
        LIMIT 5
    """, conn)
    print('Últimos datos de Santander:')
    print(muestra)


Tablas en tfg.db:
                name
0  macro_act_mensual
1   macro_act_trimes
2   macro_mon_diario
3  macro_mon_mensual
4       macro_riesgo
5    dataset_maestro
6   precios_empresas
7       ref_empresas

precios_empresas: 157,455 filas, 35 empresas

Últimos datos de Santander:
   ticker       fecha  close   log_ret  vol_hist_21d
0  SAN.MC  2025-10-31  8.826  0.008648      0.273642
1  SAN.MC  2025-10-30  8.750 -0.025946      0.279782
2  SAN.MC  2025-10-29  8.980  0.042424      0.264958
3  SAN.MC  2025-10-28  8.607  0.011216      0.223767
4  SAN.MC  2025-10-27  8.511  0.016586      0.222383


---
## Conclusiones del cuaderno ETL de Empresas

- Se han procesado las **35 empresas** del IBEX 35 con datos diarios desde 2005, las que tienen datos desde esa fecha. Si se adherieron al índice posteriormente se tienen datos de esas empresas desde la fecha de su adhesión.
- Las variables originales son: `fecha`, `close`, `net`, `pct_chg`, `open`, `low`, `high`, `volume`, `turnover_eur`
- Se han calculado dos variables derivadas: `log_ret` (log-retorno diario) y `vol_hist_21d` (volatilidad histórica anualizada a 21 días)
- Los datos están almacenados en la tabla `precios_empresas` de `tfg.db` y en formato csv como backup
